# Analyse Métriques vs Température - Pacifique

Ce notebook analyse la relation entre les métriques d'erreur du modèle (RMSE, NRMSE, MAPE) et la température de surface.

**Objectif** : Identifier si les erreurs du modèle SeapoPym DAG sont corrélées avec la température.

**Comparaisons** :

1. DAG Transport vs Référence
2. DAG No-Transport vs Référence


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

# Chemins
DATA_DIR = "./data"
FIGURES_DIR = "./figures"

# Création du dossier figures s'il n'existe pas
os.makedirs(FIGURES_DIR, exist_ok=True)

# Entrées
TEMPERATURE = os.path.join(DATA_DIR, "seapodym_lmtl_forcings_pacific.zarr")
FILE_REF = os.path.join(DATA_DIR, "seapodym_lmtl_output_pacific_ref.zarr")
FILE_TRANS = os.path.join(DATA_DIR, "seapopym_pacific_transport.zarr")
FILE_NO_TRANS = os.path.join(DATA_DIR, "seapopym_pacific_no_transport.zarr")

print(f"Ref: {FILE_REF}")
print(f"Transport: {FILE_TRANS}")
print(f"No-Transport: {FILE_NO_TRANS}")

## 1. Chargement des Données


In [ ]:
# Chargement Reference (Zooplankton + Temperature)
ds_ref = xr.open_zarr(FILE_REF)
ds_temperature = xr.open_zarr(TEMPERATURE)

ref = ds_ref["zooplankton"].load()

temp = ds_temperature["temperature"].load()

# Chargement DAG Transport
ds_trans = xr.open_zarr(FILE_TRANS)
dag_trans = ds_trans["biomass"].load()

# Chargement DAG No-Transport
ds_no_trans = xr.open_zarr(FILE_NO_TRANS)
dag_no_trans = ds_no_trans["biomass"].load()

# Standardisation des noms de coordonnées
rename_dict = {"y": "latitude", "x": "longitude", "z": "depth"}

ref = ref.rename({k: v for k, v in rename_dict.items() if k in ref.dims})
temp = temp.rename({k: v for k, v in rename_dict.items() if k in temp.dims})
dag_trans = dag_trans.rename({k: v for k, v in rename_dict.items() if k in dag_trans.dims})
dag_no_trans = dag_no_trans.rename({k: v for k, v in rename_dict.items() if k in dag_no_trans.dims})

# Sélection zone latitude [-40, 55]
ref = ref.sel(latitude=slice(-40, 55))
temp = temp.sel(latitude=slice(-40, 55))
dag_trans = dag_trans.sel(latitude=slice(-40, 55))
dag_no_trans = dag_no_trans.sel(latitude=slice(-40, 55))

print("Données chargées.")
print(f"Ref shape: {ref.shape}")
print(f"Temperature shape: {temp.shape}")
print(f"Trans shape: {dag_trans.shape}")
print(f"No-Trans shape: {dag_no_trans.shape}")

## 2. Prétraitement

Période de comparaison : 2002-2004 (après spin-up)


In [ ]:
# Période de comparaison
TIME_START = "2002-01-01"
TIME_END = "2004-12-31"

# Sélection temporelle
ref_cut = ref.sel(time=slice(TIME_START, TIME_END))
temp_cut = temp.sel(time=slice(TIME_START, TIME_END))
dag_trans_cut = dag_trans.sel(time=slice(TIME_START, TIME_END))
dag_no_trans_cut = dag_no_trans.sel(time=slice(TIME_START, TIME_END))

# IMPORTANT: Les données SeapoPym sont en pas de temps 3-horaires
# On doit les resampler à la journée pour les aligner avec la référence
print(f"Avant resampling:")
print(f"  Référence: {len(ref_cut.time)} pas de temps")
print(f"  Transport: {len(dag_trans_cut.time)} pas de temps")
print(f"  Temperature: {len(temp_cut.time)} pas de temps")

# Resampling des données SeapoPym à la journée (moyenne journalière)
dag_trans_cut = dag_trans_cut.resample(time="1D").mean()
dag_no_trans_cut = dag_no_trans_cut.resample(time="1D").mean()

print(f"\nAprès resampling à la journée:")
print(f"  Transport: {len(dag_trans_cut.time)} pas de temps")
print(f"  No-Transport: {len(dag_no_trans_cut.time)} pas de temps")

# IMPORTANT: Normalisation des timestamps à 00:00 pour tous les datasets
# La référence a des timestamps à 12:00, on doit les normaliser à 00:00
print(f"\nNormalisation des timestamps à 00:00...")
ref_cut["time"] = ref_cut.time.dt.floor("D")
temp_cut["time"] = temp_cut.time.dt.floor("D")
dag_trans_cut["time"] = dag_trans_cut.time.dt.floor("D")
dag_no_trans_cut["time"] = dag_no_trans_cut.time.dt.floor("D")

# Alignement des grilles
ref_cut, dag_trans_cut = xr.align(ref_cut, dag_trans_cut, join="inner")
ref_cut, dag_no_trans_cut = xr.align(ref_cut, dag_no_trans_cut, join="inner")
ref_cut, temp_cut = xr.align(ref_cut, temp_cut, join="inner")

# Température de surface (première couche si 4D)
temp_surf = temp_cut.isel(depth=0) if "depth" in temp_cut.dims else temp_cut

print(f"\nAprès alignement:")
print(f"  Période alignée: {len(ref_cut.time)} jours")
print(f"  Grille alignée: {len(ref_cut.latitude)} x {len(ref_cut.longitude)}")
print(f"  Temperature surface shape: {temp_surf.shape}")

## 3. Calcul RMSE, NRMSE et MAPE par Bins de Température


In [ ]:
def compute_metrics_vs_temperature(ref, model, temp, temp_bins):
    """
    Calcule RMSE, NRMSE et MAPE en fonction de bins de température.

    Parameters
    ----------
    ref : xr.DataArray
        Données de référence
    model : xr.DataArray
        Données du modèle
    temp : xr.DataArray
        Température de surface
    temp_bins : array-like
        Bins de température (ex: [0, 5, 10, 15, 20, 25, 30])

    Returns
    -------
    df : pd.DataFrame
        DataFrame avec colonnes: temp_center, rmse, nrmse, mape, n_points
    """
    # Calcul de l'erreur
    error = model - ref
    squared_error = error**2

    # Flatten des données pour faciliter le binning
    temp_flat = temp.values.flatten()
    squared_error_flat = squared_error.values.flatten()
    error_flat = error.values.flatten()
    ref_flat = ref.values.flatten()
    model_flat = model.values.flatten()

    # Masque des valeurs valides (non-NaN)
    valid_mask = (
        ~np.isnan(temp_flat)
        & ~np.isnan(squared_error_flat)
        & ~np.isnan(ref_flat)
        & ~np.isnan(model_flat)
    )

    temp_valid = temp_flat[valid_mask]
    squared_error_valid = squared_error_flat[valid_mask]
    error_valid = error_flat[valid_mask]
    ref_valid = ref_flat[valid_mask]
    model_valid = model_flat[valid_mask]

    # Binning par température
    temp_centers = []
    rmse_values = []
    nrmse_values = []
    mape_values = []
    n_points = []

    for i in range(len(temp_bins) - 1):
        t_min, t_max = temp_bins[i], temp_bins[i + 1]

        # Sélection des points dans ce bin
        mask = (temp_valid >= t_min) & (temp_valid < t_max)

        if mask.sum() > 0:
            # RMSE pour ce bin
            rmse = np.sqrt(np.mean(squared_error_valid[mask]))

            # NRMSE (normalisé par std de la référence dans ce bin)
            std_ref = np.std(ref_valid[mask])
            nrmse = (rmse / std_ref) if std_ref > 0 else np.nan

            # MAPE (pourcentage d'erreur absolue moyenne)
            ref_bin = ref_valid[mask]
            model_bin = model_valid[mask]
            # Filtrer les valeurs très petites pour éviter division par zéro
            mask_nonzero = ref_bin > 1e-6
            if mask_nonzero.sum() > 0:
                mape = (
                    np.mean(
                        np.abs(
                            (model_bin[mask_nonzero] - ref_bin[mask_nonzero])
                            / ref_bin[mask_nonzero]
                        )
                    )
                    * 100
                )
            else:
                mape = np.nan

            temp_centers.append((t_min + t_max) / 2)
            rmse_values.append(rmse)
            nrmse_values.append(nrmse)
            mape_values.append(mape)
            n_points.append(mask.sum())
        else:
            temp_centers.append((t_min + t_max) / 2)
            rmse_values.append(np.nan)
            nrmse_values.append(np.nan)
            mape_values.append(np.nan)
            n_points.append(0)

    df = pd.DataFrame(
        {
            "temp_center": temp_centers,
            "rmse": rmse_values,
            "nrmse": nrmse_values,
            "mape": mape_values,
            "n_points": n_points,
        }
    )

    return df


# Définition des bins de température (°C)
temp_bins = np.arange(0, 32, 2)  # Bins de 2°C de 0 à 30°C

print("Calcul RMSE/NRMSE/MAPE vs Température...")
print(f"Bins de température: {temp_bins}")

# Calcul pour DAG Transport vs Ref
df_trans = compute_metrics_vs_temperature(ref_cut, dag_trans_cut, temp_surf, temp_bins)
print("\n=== DAG Transport vs Référence ===")
print(df_trans)

# Calcul pour DAG No-Transport vs Ref
df_no_trans = compute_metrics_vs_temperature(ref_cut, dag_no_trans_cut, temp_surf, temp_bins)
print("\n=== DAG No-Transport vs Référence ===")
print(df_no_trans)

## 4. Visualisation


In [ ]:
# Figure avec 3 sous-plots : RMSE, NRMSE et MAPE
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- Sous-plot 1 : RMSE ---
ax1 = axes[0]
ax1.plot(
    df_trans["temp_center"],
    df_trans["rmse"],
    marker="o",
    linestyle="-",
    linewidth=2,
    markersize=6,
    label="DAG Transport",
    color="blue",
)
ax1.plot(
    df_no_trans["temp_center"],
    df_no_trans["rmse"],
    marker="s",
    linestyle="--",
    linewidth=2,
    markersize=6,
    label="DAG No-Transport",
    color="grey",
    alpha=0.7,
)

ax1.set_xlabel("Température (°C)", fontsize=12)
ax1.set_ylabel("RMSE (g/m²)", fontsize=12)
ax1.set_title("RMSE vs Température", fontsize=14, fontweight="bold")
ax1.legend(loc="best", fontsize=11)
ax1.grid(True, alpha=0.3)

# --- Sous-plot 2 : NRMSE ---
ax2 = axes[1]
ax2.plot(
    df_trans["temp_center"],
    df_trans["nrmse"],
    marker="o",
    linestyle="-",
    linewidth=2,
    markersize=6,
    label="DAG Transport",
    color="blue",
)
ax2.plot(
    df_no_trans["temp_center"],
    df_no_trans["nrmse"],
    marker="s",
    linestyle="--",
    linewidth=2,
    markersize=6,
    label="DAG No-Transport",
    color="grey",
    alpha=0.7,
)

ax2.set_xlabel("Température (°C)", fontsize=12)
ax2.set_ylabel("NRMSE", fontsize=12)
ax2.set_title("NRMSE vs Température", fontsize=14, fontweight="bold")
ax2.legend(loc="best", fontsize=11)
ax2.grid(True, alpha=0.3)

# --- Sous-plot 3 : MAPE ---
ax3 = axes[2]
ax3.plot(
    df_trans["temp_center"],
    df_trans["mape"],
    marker="o",
    linestyle="-",
    linewidth=2,
    markersize=6,
    label="DAG Transport",
    color="blue",
)
ax3.plot(
    df_no_trans["temp_center"],
    df_no_trans["mape"],
    marker="s",
    linestyle="--",
    linewidth=2,
    markersize=6,
    label="DAG No-Transport",
    color="grey",
    alpha=0.7,
)

ax3.set_xlabel("Température (°C)", fontsize=12)
ax3.set_ylabel("MAPE (%)", fontsize=12)
ax3.set_title("MAPE vs Température", fontsize=14, fontweight="bold")
ax3.legend(loc="best", fontsize=11)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(
    os.path.join(FIGURES_DIR, "fig_04_metrics_vs_temperature.png"), dpi=150, bbox_inches="tight"
)
plt.show()

## 5. Analyse par Nombre de Points


In [ ]:
# Histogramme du nombre de points par bin de température
fig, ax = plt.subplots(figsize=(12, 5))

width = 0.8
ax.bar(
    df_trans["temp_center"],
    df_trans["n_points"],
    width=width,
    alpha=0.7,
    color="steelblue",
    label="Nombre de points",
)

ax.set_xlabel("Température (°C)", fontsize=12)
ax.set_ylabel("Nombre de points", fontsize=12)
ax.set_title("Distribution des points par bin de température", fontsize=14, fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "fig_04_temp_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

print("\n=== Distribution des points par température ===")
print(df_trans[["temp_center", "n_points"]])

## 6. Export des Résultats


In [ ]:
# Export des DataFrames en CSV
df_trans.to_csv(os.path.join(FIGURES_DIR, "metrics_vs_temp_transport.csv"), index=False)
df_no_trans.to_csv(os.path.join(FIGURES_DIR, "metrics_vs_temp_no_transport.csv"), index=False)

print("Données exportées:")
print(f"  - {os.path.join(FIGURES_DIR, 'metrics_vs_temp_transport.csv')}")
print(f"  - {os.path.join(FIGURES_DIR, 'metrics_vs_temp_no_transport.csv')}")

# Résumé textuel
summary_content = f"""
================================================================================
ANALYSE MÉTRIQUES vs TEMPÉRATURE : SeapoPym DAG vs Seapodym-LMTL
================================================================================
Date: {pd.Timestamp.now()}
Période: {TIME_START} à {TIME_END}
Zone latitude: [-40, 55]

BINS DE TEMPÉRATURE :
--------------------------------------------------------------------------------
{temp_bins}

STATISTIQUES MOYENNES :
--------------------------------------------------------------------------------
DAG Transport vs Référence:
  - RMSE moyen  : {df_trans["rmse"].mean():.2f} g/m²
  - NRMSE moyen : {df_trans["nrmse"].mean():.2f}
  - MAPE moyen  : {df_trans["mape"].mean():.2f} %

DAG No-Transport vs Référence:
  - RMSE moyen  : {df_no_trans["rmse"].mean():.2f} g/m²
  - NRMSE moyen : {df_no_trans["nrmse"].mean():.2f}
  - MAPE moyen  : {df_no_trans["mape"].mean():.2f} %

FIGURES GÉNÉRÉES :
--------------------------------------------------------------------------------
- fig_04_metrics_vs_temperature.png : RMSE, NRMSE et MAPE vs Température
- fig_04_temp_distribution.png : Distribution des points par température

FICHIERS CSV :
--------------------------------------------------------------------------------
- metrics_vs_temp_transport.csv
- metrics_vs_temp_no_transport.csv

================================================================================
"""

summary_file = os.path.join(FIGURES_DIR, "metrics_vs_temperature_summary.txt")
with open(summary_file, "w") as f:
    f.write(summary_content)

print(f"\nRésumé sauvegardé dans : {summary_file}")
print(summary_content)